# 🚀 Imports

In [1]:
# Maths
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# String
import urllib
import tldextract
import re

# Misc
import sys

def print_safe(url):
    '''Prints safe, unclickable URL in VS Code.'''
    sys.stdout.write(url.replace("://", "://\x00"))  # Null byte

In [ ]:
# Number of CPU cores: 2
!lscpu

# ❗ IMPORTANT: SAFETY GUIDELINES AGAINST LINKS ⚠️

> ***READ BEFORE RUNNING ANY CELL***
>
> **The `url` column is dangerous if click link**, be careful. Other columns from are safe.
> - ⚠️ Display `df` — Always use `display(df)` so that URL are  not clickable.
> - ⚠️ Use `print_safe` — Use `print_safe(url)` so that URL are  not clickable.
> - ⚠️ String Parsing — Safe but make sure not to accidentally click link
>   - `urlparse`— Splits URL by structure
>   - `tldextract`— Splits domain specifically
>   - `re` — Finds patterns in string
> - ❌ DNS/WHOIS lookups — use `safe_dns_lookup()` only, one person runs this
> - ❌ `requests.get(url)` — never, under any circumstance


In [3]:
url = "http://mail.paypa1-secure.evil.com/login?user=1%20x"

# urlparse
from urllib.parse import urlparse
p = urlparse(url)
p.scheme    # 'http'
p.netloc    # 'mail.paypa1-secure.evil.com'
p.path      # '/login'
p.query     # 'user=1%20x'

# tldextract
ext = tldextract.extract(url)
ext.subdomain  # 'mail.paypa1-secure'
ext.domain     # 'evil'
ext.suffix     # 'com'

# re
re.search(r'\d+\.\d+\.\d+\.\d+', url)  # IP address present?
re.findall(r'\d', url)                 # digits in URL
re.search(r'%[0-9a-fA-F]{2}', url)     # hex encoding present?

print_safe(url)

http:// mail.paypa1-secure.evil.com/login?user=1%20x

---

# 🧪 1. Data Loading

In [4]:
file_path = "../data/dataset_phishing.csv"
df = pd.read_csv(file_path)
df.head()

,url,length_url,length_hostname,ip,nb_dots,nb_hyphens,nb_at,nb_qm,nb_and,nb_or,...,domain_in_title,domain_with_copyright,whois_registered_domain,domain_registration_length,domain_age,web_traffic,dns_record,google_index,page_rank,status
0,http://www.crestonwood.com/router.php,37,19,0,3,0,0,0,0,0,...,0,1,0,45,-1,0,1,1,4,legitimate
1,http://shadetreetechnology.com/V4/validation/a...,77,23,1,1,0,0,0,0,0,...,1,0,0,77,5767,0,0,1,2,phishing
2,https://support-appleld.com.secureupdate.duila...,126,50,1,4,1,0,1,2,0,...,1,0,0,14,4004,5828815,0,1,0,phishing
3,http://rgipt.ac.in,18,11,0,2,0,0,0,0,0,...,1,0,0,62,-1,107721,0,0,3,legitimate
4,http://www.iracing.com/tracks/gateway-motorspo...,55,15,0,2,2,0,0,0,0,...,0,1,0,224,8175,8725,0,0,6,legitimate


---

# 🧹 2. Data Cleaning

## A) Basic inspection of original dataset.

Note that there are many negative values within `domain_registration_length` and`domain_age` which take on values $\{-1, -2, -12\}$. From checking explanations on Kaggle, it seems to mean that **no data can be identified for it**. _This issue will be handled during parsing section_.

> Refer to [Appendix](#Appendix) to see the negative values inside the `df`

In [5]:
print("Initial Data Inspection\n" + "-" * 23)

# 1. Shape of dataset
print(f"1. Original Shape: {df.shape}")

# 2. NULL values
print(f"2. Null Values: {df.isnull().sum().sum()}") 

# 3. Negative values
print(f"3. Negative Values: {(df.iloc[:,1:-1] < 0).sum().sum()}") # exclude first and last column
print(f"    - domain_registration_length: {(df['domain_registration_length'] < 0).sum()}")
print(f"    - domain_age: {(df['domain_age'] < 0).sum()}")

# 4. Verify 'Status' column
legit_count = df[df['status'] == 'legitimate'].shape[0]
phish_count = df[df['status'] == 'phishing'].shape[0]
print(f"4. Status Column (Legitimate + Phishing): {legit_count + phish_count}")
print(f"    - Legitimate: {legit_count}")
print(f"    - Phishing: {phish_count}")
# --- More info on Status ---
# display(df['status'].value_counts())

# 5. Check for duplicate URLs
print(f"5. Duplicate Rows: {df.duplicated(subset=['url']).sum()}")
# --- More info on duplicates ---
# df[df.duplicated(subset=['url'])]
# df[df['url'] == 'http://e710z0ear.du.r.appspot.com/c:/users/user/downlo']
# print_safe(df.iloc[6035]['url'])
# print()
# print_safe(df.iloc[11305]['url'])

print("-" * 23)

Initial Data Inspection
-----------------------
1. Original Shape: (11430, 89)
2. Null Values: 0
3. Negative Values: 1883
    - domain_registration_length: 46
    - domain_age: 1837
4. Status Column (Legitimate + Phishing): 11430
    - Legitimate: 5715
    - Phishing: 5715
5. Duplicate Rows: 1
-----------------------


## B) Patching to new dataset `df_clean`.

_Techniques used to handle the negative values_ for `domain_registration_length` and `domain_age` respectively are:
- Deletion — 46 values are little (0.42% of dataset) 
- Imputation — 1800 values are large (16.36% of dataset)

Therefore, the following will be carried out on the dataset:
1. **Deletion** of rows with negative values in `domain_registration_length`.
2. An **additional column** called `domain_age_missing` will be added as potential feature.
3. For **imputation**, we shall use **median values** because the range for `domain_age` can have outliers.
4. **Binary encoding** of `[legitimate,phishing]` in `status` column to `[0,1]` respectively.
5. **Deletion** of duplicate rows.

In [6]:
# Some calculations & variables
numeric_cols = df.select_dtypes(include=[np.number]).columns # temp variable
domain_age_median = df[df['domain_age'] >= 0]['domain_age'].median()

# New cleaned df
df_clean = df.copy() 

# ----- Data Cleaning Steps -----

# 1) Remove rows with negative values for 'domain_registration_length' 
df_clean = df_clean[df_clean['domain_registration_length'] >= 0]

# 2) Create missing indicator column beside 'domain_age'
#    Create missing indicator column
df_clean['domain_age_missing'] = np.where(df_clean['domain_age'] < 0, 1, 0)
#    Move it right beside 'domain_age'
df_clean.insert(df_clean.columns.get_loc('domain_age') + 1, 'domain_age_missing', df_clean.pop('domain_age_missing'))

# 3) Impute median value replacing where negative for 'domain_age'
df_clean['domain_age'] = df_clean['domain_age'].where(df_clean['domain_age'] >= 0, domain_age_median)

# 4. Binary encoding for 'status' column
df_clean['status'] = np.where(df_clean['status'] == 'legitimate', 0, 1)

# 5. Remove duplicate URLs
df_clean = df_clean.drop_duplicates(subset=['url'])

Sanity check on cleaned data.

In [16]:
print("Clean Data Inspection\n" + "-" * 23)

# 1. Shape of dataset
print(f"1. Original Shape: {df_clean.shape}")

# 2. NULL values
print(f"2. Null Values: {df_clean.isnull().sum().sum()}") 

# 3. Negative values
print(f"3. Negative Values: {(df_clean.iloc[:,1:-1] < 0).sum().sum()}") # exclude first and last column
print(f"    - domain_registration_length: {(df_clean['domain_registration_length'] < 0).sum()}")
print(f"    - domain_age: {(df_clean['domain_age'] < 0).sum()}")

# 4. Verify 'Status' column
legit_count = df_clean[df_clean['status'] == 0].shape[0]
phish_count = df_clean[df_clean['status'] == 1].shape[0]
total_count = legit_count + phish_count
print(f"4. Status Column (Legitimate + Phishing): {legit_count + phish_count}")
print(f"    - Legitimate: {legit_count} ({legit_count/total_count:.2%})")
print(f"    - Phishing: {phish_count} ({phish_count/total_count:.2%})")
# --- More info on Status ---
# display(df_clean['status'].value_counts())

# 5. Check for duplicate URLs
print(f"5. Duplicate Rows: {df_clean.duplicated(subset=['url']).sum()}")
# --- More info on duplicates ---
# df_clean[df_clean.duplicated(subset=['url'])]
# df_clean[df_clean['url'] == 'http://e710z0ear.du.r.appspot.com/c:/users/user/downlo']
# print_safe(df_clean.iloc[6035]['url'])
# print()
# print_safe(df_clean.iloc[11305]['url'])

print("-" * 23)

# df_clean.head()
df_clean.tail()

Clean Data Inspection
-----------------------
1. Original Shape: (11383, 90)
2. Null Values: 0
3. Negative Values: 0
    - domain_registration_length: 0
    - domain_age: 0
4. Status Column (Legitimate + Phishing): 11383
    - Legitimate: 5683 (49.93%)
    - Phishing: 5700 (50.07%)
5. Duplicate Rows: 0
-----------------------


,url,length_url,length_hostname,ip,nb_dots,nb_hyphens,nb_at,nb_qm,nb_and,nb_or,...,domain_with_copyright,whois_registered_domain,domain_registration_length,domain_age,domain_age_missing,web_traffic,dns_record,google_index,page_rank,status
11425,http://www.fontspace.com/category/blackletter,45,17,0,2,0,0,0,0,0,...,0,0,448,5396,0,3980,0,0,6,0
11426,http://www.budgetbots.com/server.php/Server%20...,84,18,0,5,0,1,1,0,0,...,0,0,211,6728,0,0,0,1,0,1
11427,https://www.facebook.com/Interactive-Televisio...,105,16,1,2,6,0,1,0,0,...,0,0,2809,8515,0,8,0,1,10,0
11428,http://www.mypublicdomainpictures.com/,38,30,0,2,0,0,0,0,0,...,0,0,85,2836,0,2455493,0,0,4,0
11429,http://174.139.46.123/ap/signin?openid.pape.ma...,477,14,1,24,0,1,1,9,0,...,1,1,0,5076,1,0,1,1,0,1


---

## Appendix

Some checks to visually verify unclean data.

In [8]:
# Find columns with negative values
numeric_cols = df.select_dtypes(include=[np.number]).columns
neg_cols = [col for col in numeric_cols if (df[col] < 0).any()]

print(f"Columns with negatives ({len(neg_cols)}): {neg_cols}")

# Find unique negative values
neg_vals = pd.concat([df[col] for col in neg_cols]).unique()
neg_vals = neg_vals[neg_vals < 0]

print(f"Unique negative values ({len(neg_vals)}): {sorted(neg_vals)}")

Columns with negatives (2): ['domain_registration_length', 'domain_age']
Unique negative values (3): [np.int64(-12), np.int64(-2), np.int64(-1)]


In [9]:
# Show rows with negative domain_registration_length
df[df['domain_registration_length'] < 0][['url', 'domain_registration_length']].head()

,url,domain_registration_length
443,https://www.hfunderground.com/wiki/Ionosphere,-1
776,http://resources.infosecinstitute.com/applicat...,-1
870,http://www.abm.edu.ar/,-1
1092,https://www.maria-black.com/en/eu/,-1
1208,http://gryffilm.wz.cz/8CBF35A7E54771A2B27F0FB9...,-1


In [10]:
# Show rows with negative domain_age
df[df['domain_age'] == -2][['url', 'domain_age']].head()

,url,domain_age
200,https://mauy.org/PH/docusign/DocuSign/u.php?l=...,-2
307,https://vktheme.ru/auth,-2
613,http://ddrevent.com/board/wp-content/themes/al...,-2
659,http://teksty-pesenok.ru/tracey-curtis/tekst-p...,-2
971,http://speedonline.fr/wp-content/plugins/index...,-2


In [11]:
# Show rows with negative domain_age
df[df['domain_age'] == -12][['url', 'domain_age']].head()

,url,domain_age
6514,http://fastfoodgozo.rs/a/yahoo/login.php,-12
